# 配对交易项目：从结果读懂代码
先阅读 README_中文.md。此 notebook 读取已完成的真实回测；不重新挑参数。

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
root = Path.cwd()
out = root / "results" / "real_data"
meta = json.loads((out / "run_metadata.json").read_text())
meta

## 1. 形成期筛股票
p 值小表示反对“不协整”的证据更强。q 值是多重检验调整结果。这里不能根据测试收益改股票对。

In [ ]:
pairs = pd.read_csv(out / "cointegration_tests.csv")
pairs.head(20)

## 2. 验证期选参数
这里按净 Sharpe 排序，但最终选择还要求足够的交易数。配置保存在 config.json。

In [ ]:
grid = pd.read_csv(out / "validation_sensitivity.csv")
grid.sort_values("sharpe", ascending=False).head(15)

## 3. 看最终测试
年化收益回答赚多少；波动率与最大回撤描述风险；Sharpe 把平均收益与波动结合起来。

In [ ]:
display(pd.read_csv(out / "performance.csv"))
display(Image(filename=str(out / "equity_curve.png")))

## 4. 查每一笔交易与成本
胜率高也可能亏钱，必须同时看每笔盈亏大小。

In [ ]:
trades = pd.read_csv(out / "test_trades.csv")
display(trades.head(20))
display(pd.read_csv(out / "test_cost_stress.csv"))

## 5. 回到代码
阅读顺序：load_prices → screen → choose_pairs → signals → backtest → evaluate → metrics → main。先理解 signals 的 alpha、beta、z，再看延迟一日的成交规则。

In [ ]:
import inspect
from project import signals, backtest
print(inspect.getsource(signals))